# L15 – Exempellösning: California Housing, MLP mot gradient boosting

Här är ett komplett exempel på arbetsflödet, steg för steg. Det är ett **regressionsproblem på tabelldata**, valt för att visa att arbetsflödet är detsamma även när problemet är ett annat. Det visar också en ärlig slutsats: det neurala nätet är inte alltid bäst.

In [ ]:
import sys
sys.path.insert(0, "..")
import copy, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from dlkit.seed import set_seed

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 0–1. Problem och mått
- **Problem:** förutsäg medianpriset för bostäder i ett distrikt (i enheten 100 000 dollar) från 8 features: inkomst, husålder, antal rum, läge och så vidare.
- **Mått:** **RMSE**, eftersom stora felbedömningar av priset är särskilt dåliga. **MAE** rapporteras också, eftersom den är lätt att tolka ("i genomsnitt fel med X dollar").

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.shape)
print(df.describe().T[["mean", "std", "min", "max"]].round(2))

`AveRooms`, `AveBedrms` och `AveOccup` har extrema maxvärden (outliers), och målet är kapat vid 5,0. Det är värt att ha i åtanke. I L03 såg vi att outliers kan ge stora steg med små batcher.

## 2. Uppdelning
Slumpvis uppdelning: 60 % train, 20 % val och 20 % test. Distrikten är oberoende och det finns ingen tidsordning.

In [ ]:
X, y = data.data.to_numpy(dtype="float32"), data.target.to_numpy(dtype="float32")
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, random_state=42)
print(len(X_train), len(X_val), len(X_test))

def report(name, y_true, y_pred):
    return {"modell": name, "RMSE": root_mean_squared_error(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred)}

## 3. Baslinjer

In [ ]:
scaler = StandardScaler().fit(X_train)          # bara train
X_train_s, X_val_s, X_test_s = (scaler.transform(a).astype("float32") for a in (X_train, X_val, X_test))

results = [report("medel (naiv)", y_val, np.full_like(y_val, y_train.mean())),
           report("LinearRegression", y_val, LinearRegression().fit(X_train_s, y_train).predict(X_val_s))]
gbm = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
results.append(report("HistGradientBoosting", y_val, gbm.predict(X_val)))
pd.DataFrame(results).round(4)

Gradient boosting är en **mycket stark** baslinje för tabelldata. Det blir intressant att se om en MLP kan slå den.

## 4. Datapipeline
- Standardisering (redan gjord ovan, anpassad på train).
- Ingen augmentation, eftersom det inte finns någon naturlig sådan för tabelldata.
- Batch 128.

In [ ]:
def loader(Xa, ya, shuffle, bs=128):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya)[:, None]), batch_size=bs, shuffle=shuffle)

train_loader = loader(X_train_s, y_train, True)
val_loader = loader(X_val_s, y_val, False, 1024)
test_loader = loader(X_test_s, y_test, False, 1024)

## 5. Modell och sanity checks
- **En MLP**, eftersom datan är tabelldata utan rumslig struktur eller ordning (L05).
- **8 in och 1 ut**, utan aktivering i sista lagret: regression ger ett obegränsat tal.
- **ReLU** i de dolda lagren.
- **MSELoss**, i linje med valet av RMSE.

In [ ]:
def make_mlp(hidden=128, dropout=0.0):
    return nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(hidden // 2, 1))

loss_fn = nn.MSELoss()
set_seed(42)
m = make_mlp().to(device)
Xb, yb = next(iter(train_loader))
Xb, yb = Xb.to(device), yb.to(device)
with torch.no_grad():
    print(f"loss vid start: {loss_fn(m(Xb), yb).item():.3f}  (om modellen gissar ≈0 blir det ≈ medel(y²) = {(yb**2).mean().item():.3f})")

opt = torch.optim.Adam(m.parameters(), lr=1e-3)
for _ in range(1000):
    opt.zero_grad(); loss = loss_fn(m(Xb), yb); loss.backward(); opt.step()
print(f"överanpassning av en batch: loss {loss.item():.4f}")
assert loss.item() < 0.02, "modellen kan inte lära sig en enda batch – leta efter en bugg"

För regression är $\ln K$ inte relevant. I stället ska en otränad modell som gissar nära 0 ge MSE ≈ medelvärdet av $y^2$. Båda kontrollerna är godkända.

## 6. Träning med loggning
`dlkit.fit` räknar accuracy, så för regression skriver vi en kort loop med early stopping (L07) och scheduler (L08).

In [ ]:
def train_regressor(model, lr=1e-3, weight_decay=0.0, epochs=100, patience=10, verbose=False):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=4)
    hist, best, best_state, bad = {"train": [], "val": []}, float("inf"), None, 0
    for epoch in range(epochs):
        model.train(); tot = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(Xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(yb)
        val_mse = evaluate_mse(model, val_loader)
        hist["train"].append(tot / len(train_loader.dataset)); hist["val"].append(val_mse)
        sched.step(val_mse)
        if val_mse < best:
            best, best_state, bad = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    if verbose:
        print(f"stoppade efter {len(hist['val'])} epoker, bästa val RMSE {best ** 0.5:.4f}")
    return hist

@torch.no_grad()
def evaluate_mse(model, loader_):
    model.eval()
    return sum(loss_fn(model(Xb.to(device)), yb.to(device)).item() * len(yb) for Xb, yb in loader_) / len(loader_.dataset)

@torch.no_grad()
def predict(model, Xa):
    model.eval()
    return model(torch.tensor(Xa, device=device)).cpu().numpy()[:, 0]

set_seed(42)
mlp = make_mlp()
t0 = time.time()
hist = train_regressor(mlp, verbose=True)
print(f"{time.time() - t0:.0f} s")
plt.figure(figsize=(7, 3.5))
plt.plot(np.sqrt(hist["train"]), label="train RMSE"); plt.plot(np.sqrt(hist["val"]), label="val RMSE")
plt.xlabel("epok"); plt.legend(); plt.grid(alpha=0.3); plt.show()
results.append(report("MLP 128-64", y_val, predict(mlp, X_val_s)))
pd.DataFrame(results).round(4)

## 7. Diagnos
Kurvorna visar att train och val sjunker tillsammans och planar ut, med ett litet gap. Det är **ingen tydlig överanpassning**. MLP:n är klart bättre än linjär regression men inte bättre än gradient boosting.

**Felanalys:** var gör modellen störst fel?

In [ ]:
val_pred = predict(mlp, X_val_s)
err = np.abs(val_pred - y_val)
bins = pd.cut(y_val, [0, 1, 2, 3, 4, 4.99, 5.01])
print(pd.DataFrame({"pris": bins, "abs fel": err}).groupby("pris", observed=True)["abs fel"].agg(["mean", "count"]).round(3))

Felen är störst för de **dyraste distrikten**. Målet är kapat vid 5,0, så "500 000 dollar eller mer" är samma värde. Det är en egenhet i datan som ingen modell kan komma runt helt.

## 8. Förbättringar: en liten random search

In [ ]:
rng = np.random.default_rng(0)
trials = []
for i in range(8):
    cfg = {"lr": float(10 ** rng.uniform(-3.5, -2)), "weight_decay": float(10 ** rng.uniform(-4, -1)),
           "hidden": int(rng.choice([64, 128, 256, 512])), "dropout": float(rng.uniform(0, 0.3))}
    set_seed(42)
    model = make_mlp(cfg["hidden"], cfg["dropout"])
    h = train_regressor(model, cfg["lr"], cfg["weight_decay"])
    trials.append({**cfg, "val_RMSE": min(h["val"]) ** 0.5, "epoker": len(h["val"])})
search = pd.DataFrame(trials).sort_values("val_RMSE")
search.round(4)

In [ ]:
best_cfg = search.iloc[0].to_dict()
set_seed(42)
best_mlp = make_mlp(int(best_cfg["hidden"]), best_cfg["dropout"])
train_regressor(best_mlp, best_cfg["lr"], best_cfg["weight_decay"])
results.append(report("MLP (bästa från sökning)", y_val, predict(best_mlp, X_val_s)))
pd.DataFrame(results).round(4)

## 9. Slutlig utvärdering på test – en gång
Valet görs på **val**. Gradient boosting hade lägst val RMSE, så det är den modell vi skulle välja. För att få en fullständig rapport utvärderar vi både den och den bästa MLP:n, en gång.

In [ ]:
final = pd.DataFrame([report("HistGradientBoosting", y_test, gbm.predict(X_test)),
                      report("MLP (bästa)", y_test, predict(best_mlp, X_test_s))]).round(4)
final

## 10. Spara

In [ ]:
ckpt_dir = Path("../checkpoints"); ckpt_dir.mkdir(exist_ok=True)
torch.save(best_mlp.state_dict(), ckpt_dir / "california_mlp.pt")
(ckpt_dir / "california_mlp_config.json").write_text(json.dumps(
    {**{k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in best_cfg.items()},
     "scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist()}, indent=2))
print("sparat:", sorted(p.name for p in ckpt_dir.glob("california_mlp*")))

Skalningens parametrar sparas **tillsammans med modellen**. Utan dem går det inte att använda modellen på ny data, eftersom indata måste förbehandlas exakt som under träningen (L04).

## Rapport
1. **Problem och mått:** regression av bostadspriser. RMSE är huvudmått och MAE kompletterar.
2. **Resultat:** se tabellerna ovan. MLP:n slår linjär regression tydligt men når inte riktigt upp till gradient boosting, varken på val eller på test.
3. **Viktigaste val:** MLP med ReLU och MSELoss, standardisering på train, AdamW med early stopping och ReduceLROnPlateau, och random search över learning rate, bredd, weight decay och dropout.
4. **Vad fungerade inte?** Sökningen förbättrade MLP:n märkbart (val RMSE från cirka 0,54 till 0,50), men det räckte inte för att slå gradient boosting. Felen domineras av de dyraste distrikten, där målet är kapat.
5. **Nästa steg:** feature engineering (till exempel avstånd till kusten från lat/long), en ensemble av MLP och GBM, eller att hantera det kapade målet separat.

**Lärdom:** för tabelldata är gradient boosting ofta svårslaget. Neurala nät är överlägsna när datan har **struktur**: bilder (CNN), sekvenser och text (RNN och transformers), eller när förtränade modeller kan användas. Arbetsflödet, med baslinje, val och ärlig testning, är det som visar vilket verktyg som passar.